# Занятие 1. Агент как цикл

**К концу пары у вас будет работающий агент без фреймворка.**

- цикл `for`, текстовый парсер и одна обычная функция на Python
- контракт курса описан в отдельном документе


## Модель как компонент

**Чат-модель — это функция, которая принимает список сообщений и возвращает одно сообщение.**

- сегодня из API нужны только три роли: `system`, `user` и `assistant`
- сегодня без оберток: код работает с клиентом провайдера напрямую
- `MODEL_CHEAP` сегодня, `MODEL_STRONG` с занятия 2, оба из `.env`
- gpt-5 отключат 11 декабря, во время семестра; у Gemini 2.5 была дата отключения 16 октября, потом ее сняли


In [ ]:
# Собрать клиент провайдера из трех переменных .env, чтобы ни ключа, ни имени модели не было в коде.

import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # читает .env отсюда или из любой папки выше

API_KEY = os.environ["LLM_API_KEY"]
BASE_URL = os.environ["LLM_BASE_URL"]  # неверный путь в адресе дает 404, будто дело в ключе
MODEL = os.environ["MODEL_CHEAP"]      # модели умирают за семестр: правьте .env, не код

# gpt-5.x в chat completions: инструменты работают только с выключенными рассуждениями
EXTRA = {"reasoning_effort": os.environ["LLM_REASONING_EFFORT"]} if os.getenv("LLM_REASONING_EFFORT") else {}
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=120)  # каждая попытка обрывается через 2 минуты вместо 10 по умолчанию
print("endpoint:", BASE_URL, "| model:", MODEL)


In [ ]:
# Отправить один запрос с сообщениями system и user, чтобы увидеть объект ответа, finish_reason и usage.

answer = client.chat.completions.create(
    model=MODEL,
    messages=[  # все состояние разговора; оно уходит заново при каждом вызове
        {"role": "system", "content": "You are terse. Two sentences at most."},  # всегда
        {"role": "user", "content": "Name one book about graph algorithms."},  # ваш ввод
    ],
    max_completion_tokens=2000,  # скрытые рассуждения тоже тратят этот бюджет
)

print(answer.choices[0].message)  # choices — список; в message есть и tool_calls
print()
print("finish_reason:", answer.choices[0].finish_reason)  # stop или кончился бюджет
print("usage:        ", answer.usage)


In [ ]:
# Сделать три отдельных вызова, два одинаковых и один уточняющий: общей памяти у вызовов нет.

def ask(messages):
    reply = client.chat.completions.create(
        model=MODEL, messages=messages, max_completion_tokens=2000
    )
    return reply.choices[0].message.content  # только текст: объект целиком был выше


Q = "Name one book about graph algorithms."

# одинаковые ответы здесь означали бы детерминизм, а не память
print("call 1:", ask([{"role": "user", "content": Q}]))
print()
print("call 2:", ask([{"role": "user", "content": Q}]))
print()
print("call 3:", ask([{"role": "user", "content": "Who wrote it?"}]))


**Новый вызов не видит прошлых.**

- вызов 3 спрашивает, кто написал *it*; в этом вызове не сказано, что такое *it*, а модель все равно отвечает


In [ ]:
# Собрать историю руками в список из трех сообщений, чтобы уточняющему вопросу было к чему относиться.

first = ask([{"role": "user", "content": Q}])  # один настоящий вызов, дальше подделка

conversation = [
    {"role": "user", "content": Q},
    {"role": "assistant", "content": first},
    {"role": "user", "content": "Who wrote it?"},
]

print(ask(conversation))  # снова вызов 3: теперь у *it* есть, к чему относиться


**Ваш код хранит разговор в списке и отправляет его целиком при каждом вызове.**

- провайдер между вызовами ничего не хранит
- баги памяти: не тот список, перезаписанный список, копия вместо оригинала
- занятия 2-16 будут наращивать цикл, но разговор останется списком
- курс считает модель черным ящиком и учит работать с ее интерфейсом; как она устроена внутри: https://karpathy.ai/zero-to-hero.html


## Каналы доступа

**Провайдеру уходит и вывод инструментов, поэтому данные в инструментах только выдуманные.**

- таблица каналов есть в предкурсовом документе
- на бесплатном тарифе считайте, что провайдер хранит отправленное и что это читают люди


## Цикл

**Инструмент — обычная функция поиска по шести книгам, и при промахе она дает подсказку.**

- без декоратора и базового класса
- простой поиск по подстроке, выбран нарочно, чтобы он мог промахнуться


In [ ]:
# Написать инструмент: поиск на обычном Python по шести книгам; промах подсказывает модели, что делать.

BOOKS = [
    ("Introduction to Algorithms", "algorithms, data structures, graphs"),
    ("The Pragmatic Programmer", "craft, habits, career"),
    ("Designing Data-Intensive Applications", "databases, distributed systems"),
    ("Structure and Interpretation of Computer Programs", "recursion, interpreters"),
    ("Refactoring", "code smells, tests, design"),
    ("The Mythical Man-Month", "teams, estimation, schedules"),
]


def find_book(query: str) -> str:  # здесь ничто не знает, что существует модель
    """Search the catalogue by title or topic. One short word works best."""
    needle = query.strip().lower()
    hits = [t for t, topics in BOOKS if needle in t.lower() or needle in topics]
    if not hits:
        # для модели, а не для лога: получив None, она сдастся
        return f"No match for {query!r}. Try a single keyword, for example: graphs, tests, databases."
    return "; ".join(hits)


print(find_book("graphs"))
print(find_book("recursion"))
print(find_book("something about graph algorithms"))  # с такого запроса модель и начнет


**На следующем витке модель знает о промахе только то, что сказано в этом сообщении.**

- занятие 3 почти целиком про то, как написать это одно предложение


### Системный промпт

**Системный промпт задает весь протокол; прочитайте его вслух.**

- формат ответа задан заранее, поэтому вызов находит `str.startswith`


In [ ]:
# Зафиксировать формат ответа в системном промпте: вызов инструмента станет текстом, который найдет парсер.

# модель выдает текст, значит, и вызов инструмента — текст
SYSTEM = """You answer questions using a book catalogue.

Work in a loop and use exactly this format:

Thought: what you need to find out
Action: find_book("<query>")
Observation: <what the tool returned>

When you know the answer, reply with:

Final Answer: <your answer>
"""

# первый предохранитель в промпте, второй — обрезка ниже
STOP_RULE = "\nAfter the Action line STOP and wait: never write the Observation yourself."

# фраза, а не ключевое слово: первый вызов промахнется
QUESTION = "Can you recommend me something about graph algorithms?"

print(SYSTEM)


### Обрезка ответа

**Модель может нарушить «STOP and wait», поэтому код обрезает каждый ответ по «Observation:».**

- нынешние модели обычно слушаются, слабые сами пишут Observation
- обрезка занимает три строки кода и работает на любой модели, соблюдает та правило или нет
- ReAct-парсер из LangChain — регулярное выражение по `Action:` / `Action Input:`, рассчитанное на стоп-слово API
- без этого стоп-слова выдуманный Observation попадает внутрь аргумента инструмента

In [ ]:
# Обрезать ответ в коде перед Observation: правило STOP само по себе — лишь просьба к модели.

# параметр stop на моделях с рассуждениями дает 400, поэтому обрезка здесь
def cut_at_observation(text: str) -> str:
    marker = text.find("Observation:")
    return text if marker == -1 else text[:marker].rstrip()  # -1: резать нечего


# необрезанный ответ: модель сама ответила на свой вызов инструмента
FAKE = """Thought: I should look this up.
Action: find_book("graphs")
Observation: Introduction to Algorithms, third edition.
Final Answer: Read Introduction to Algorithms."""

print(cut_at_observation(FAKE))


In [ ]:
# Разобрать строку Action через startswith и срез; когда модель закончила, вернуть None.

def parse_action(text: str) -> str | None:
    """Pull the query out of: Action: find_book("graphs")"""
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("Action:") and "(" in line:  # взять то, что в скобках
            return line[line.index("(") + 1 : line.rindex(")")].strip().strip("\"'")
    return None  # готово: единственный сигнал выхода из цикла


print(repr(parse_action(FAKE)))
print(repr(parse_action("Final Answer: Introduction to Algorithms.")))


### Как вернуть результат

**Наблюдение возвращается сообщением `user`, потому что API не продолжает последнее сообщение `assistant`.**

- в курсе Hugging Face его дописывают в сообщение ассистента; с реальными провайдерами это не работает
- в витке шесть этапов: отправить список, модель пишет ответ, обрезать, найти Action, запустить инструмент, дописать два сообщения
- модель выполняет один этап из шести, остальные пять выполняет код


In [ ]:
# Собрать агента: ограниченный цикл for шлет, обрезает, разбирает, вызывает инструмент и возвращает результат.

def run(honest: bool = True, steps: int = 5) -> list[str]:
    messages = [  # honest=False снимает оба предохранителя: правило и обрезку
        {"role": "system", "content": SYSTEM + (STOP_RULE if honest else "")},
        {"role": "user", "content": QUESTION},
    ]
    transcript = [f"Question: {QUESTION}"]  # расшифровка прогона, ее вы и закоммитите

    for _ in range(steps):
        answer = client.chat.completions.create(
            model=MODEL,
            messages=messages,  # уходит целиком: этот список и есть вся память
            max_completion_tokens=2000,
        )
        reply = answer.choices[0].message.content or ""  # content бывает None, а не ""
        if honest:
            reply = cut_at_observation(reply)
        transcript.append(reply)
        print(reply)

        query = parse_action(reply)
        if query is None:  # строки Action нет: модель закончила или так думает
            break

        observation = f"Observation: {find_book(query)}"  # здесь и правда работает Python
        transcript.append(observation)
        print(observation)

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": observation})  # user, а не assistant

    return transcript


**Ограничивайте каждый цикл, который пишете в этом семестре.**

- на занятии 2 предел задает `recursion_limit`: он считает запуски узлов, около двух на виток; в langgraph 1.2 по умолчанию 10007, а не 25


In [ ]:
# Запустить честный цикл один раз и посчитать строки Action: вторая исправляет промах.

transcript = run()  # посчитайте строки Action: их должно быть не меньше двух


**По фразе ничего не нашлось, и в следующем Action модель взяла слово из подсказки.**

- прогоны различаются: две или три строки Action; смотрите на вторую
- ответ модели и результат инструмента изменили то, что код отправил дальше; этот цикл и есть агент

### Теперь без обоих предохранителей


In [ ]:
# Запустить тот же цикл без обоих предохранителей и посмотреть, как модель выдумывает свой Observation.

broken = run(honest=False)  # в промпте нет правила STOP, в коде нет обрезки


**Ни исключения, ни предупреждения: формат соблюден, а книги выдуманы.**

- во всех записанных прогонах модель пишет свой Observation до того, как запустился `find_book`
- настоящий Observation приходит позже; модель отвечает новым Action и еще одним выдуманным Observation
- случайно верный ответ опасен, ведь именно такую версию вы и выпустите
- обрезка работает на любом адресе из таблицы каналов, соблюдает модель правило или нет


### Две поломки, в которых винят парсер

**При бюджете 16 токенов ответ пуст или обрывается до Action, а `finish_reason` равен `length`.**

- с рассуждениями все 16 токенов уходят на скрытые рассуждения, и ответ пуст
- без рассуждений ответ — видимый текст, который обрывается до полной строки Action
- у OpenAI в некоторые дни такой запрос вместо этого дает ошибку 400: причина та же, исход третий

In [ ]:
# Отправить тот же запрос с бюджетом 16 токенов и увидеть сбой, похожий на ошибку парсера.

from openai import BadRequestError

try:
    tiny = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": QUESTION},
        ],
        max_completion_tokens=16,  # нарочно мало: скрытые рассуждения тоже тратят бюджет
    )
    print("content:      ", repr(tiny.choices[0].message.content))
    print("finish_reason:", tiny.choices[0].finish_reason)  # length, и ничего не упало
    print("usage:        ", tiny.usage)
except BadRequestError as error:  # тот же малый бюджет, другой исход: в некоторые дни это 400
    print("400 from the provider:", error.body["error"]["message"])

In [ ]:
# Отдать парсеру ответ, оборванный посреди Thought: вернется None, как и для готового ответа.

truncated = "Thought: I should search the cat"

print("content:      ", repr(truncated))
print("parse_action: ", repr(parse_action(truncated)))  # None, как у готового ответа


**На ответе, оборванном из-за бюджета, парсер верно возвращает `None`, и цикл заканчивается.**

- `None` значит «готово», поэтому пустой или недописанный ответ уходит как окончательный
- `run()` в скелете практики бросает исключение с понятным текстом, но только на пустом ответе
- сначала проверьте `finish_reason` и `usage`, потом парсер


## Тот же запрос со схемой

**Один новый параметр: какие из пяти этапов кода останутся?**

- `tools=` передает формат вызова JSON-схемой, и из системного промпта его можно убрать
- провайдер возвращает вызов уже разобранным, объектом
- сегодня только один запрос: второй виток на совместимом адресе Gemini дает ошибку 400

In [ ]:
# Описать тот же инструмент JSON-схемой: половина SYSTEM про формат переезжает в запрос.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "find_book",  # имя, которое модель напечатает в ответ
            # описание и пояснения к параметрам — это промпт, а не документация
            "description": "Search the book catalogue. One short keyword works best.",
            "parameters": {  # JSON Schema, в ней половина SYSTEM про формат
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "A single keyword"}
                },
                "required": ["query"],
            },
        },
    }
]


In [ ]:
# Отправить запрос с tools= и без системного промпта, чтобы провайдер вернул вызов объектом.

import json

native = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],  # системного промпта нет вообще
    tools=TOOLS,  # единственное, что добавилось к самому первому запросу
    max_completion_tokens=2000,
    **EXTRA,  # у большинства провайдеров пусто, см. ячейку с настройкой
)
message = native.choices[0].message

print("content:   ", repr(message.content))
print("tool_calls:", message.tool_calls)

# только один запрос: второй виток на совместимом адресе Gemini дает 400
for call in message.tool_calls or []:
    arguments = json.loads(call.function.arguments)
    print(f"\ncalling {call.function.name}({arguments}) ->")
    print(find_book(**arguments))


**Со схемой ответ разбирает провайдер, а инструмент по-прежнему запускает ваш код.**

- модель по-прежнему пишет текст; в объект его превращает провайдер
- из кода ушли промпт с форматом, обрезка и `parse_action`
- в коде остались отправка списка, запуск функции и дописывание результата
- аргументы приходят строкой с JSON, поэтому `json.loads` остается
- на занятии 2 сам цикл выполняет LangGraph

## Практика

**Свой репозиторий, с нуля.**

- `git init`, `.env` с четырьмя переменными, один файл на Python
- `.env` — в `.gitignore` до первого коммита: запушенный ключ считают утекшим
- четыре TODO по порядку; после каждого файл должен запускаться
- ваш `parse_action` должен находить формат, который задает ваш промпт
- этот файл вы будете наращивать каждое занятие до декабря

```python
import os
import sys

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
MODEL = os.environ["MODEL_CHEAP"]

# TODO 1. Имя инструмента, формат вызова; правило STOP остается отдельно
SYSTEM = """TODO"""
STOP_RULE = "\nAfter the Action line STOP and wait: never write the Observation yourself."
QUESTION = "TODO"


def my_tool(argument: str) -> str:
    """TODO 2. Строка на входе, строка на выходе. Переименуйте функцию по смыслу и
    оставьте TOOL указывающим на нее. Ветка промаха должна говорить, каким был бы
    аргумент получше: кроме этой фразы, у модели ничего нет."""
    raise NotImplementedError


TOOL = my_tool


def parse_action(text: str) -> str | None:
    """TODO 3. Найдите вызов, верните аргумент, верните None, когда модель
    закончила. Хватит построчного разбора через str.startswith."""
    raise NotImplementedError


def cut_at_observation(text: str) -> str:
    marker = text.find("Observation:")
    return text if marker == -1 else text[:marker].rstrip()


def run(honest: bool = True, steps: int = 5) -> list[str]:
    messages = [{"role": "system", "content": SYSTEM + (STOP_RULE if honest else "")},
                {"role": "user", "content": QUESTION}]
    transcript = [f"Question: {QUESTION}"]
    for _ in range(steps):
        answer = client.chat.completions.create(
            # заранее с запасом: скрытые рассуждения тоже тратят этот бюджет
            model=MODEL, messages=messages, max_completion_tokens=2000
        )
        choice = answer.choices[0]
        reply = choice.message.content or ""
        # это баг бюджета, а не парсера: назовите его
        if not reply and choice.finish_reason == "length":
            raise RuntimeError(
                "Empty reply at the length limit: the budget went on hidden "
                "reasoning, not on your parser. Raise max_completion_tokens."
            )
        if honest:
            reply = cut_at_observation(reply)
        transcript.append(reply)
        print(reply)

        argument = parse_action(reply)
        if argument is None:
            break
        observation = f"Observation: {TOOL(argument)}"
        transcript.append(observation)
        print(observation)
        messages.append({"role": "assistant", "content": reply})
        # TODO 4. Верните наблюдение: одна строка. Чей это ход?
    return transcript


if __name__ == "__main__":
    # --break снимает оба предохранителя: вторая расшифровка для коммита
    run(honest="--break" not in sys.argv)
```


**Ваш инструмент — чистый Python без сети, на тему, которой нет в демонстрациях курса.**

- без сети и ключей: вы отлаживаете свой цикл, а не чужой API
- заняты демонстрациями курса: поиск по книгам, уход за растениями, курсы валют, обращения в поддержку
- список тем для примера есть в предкурсовом документе


**Обязательный артефакт: две расшифровки в `runs/session-01.md`, в коммите.**

- запустите файл дважды: без флага и с `--break`
- обе расшифровки положите в `runs/session-01.md`, сверху имя и сигнатура инструмента
- перед коммитом верните оба предохранителя в код
- одно предложение своими словами о том, что показал второй прогон


**Если закончили раньше.**

- настоящий инструмент вместо таблицы, с полезным сообщением о промахе
- на занятии 5 выбирают проект: лучше, чтобы написанное вам нравилось


## Сегодня в одной карточке

**Агент — это цикл, который подает модели на вход ее ответ и результат инструмента.**

**Теперь вы можете объяснить:**
- разговор — ваш список, он уходит целиком: провайдер между вызовами ничего не хранит
- вызов инструмента — это текст; со схемой его разбирает провайдер, а запускает ваш код
- модель может обойти «STOP and wait», но не обрезку в вашем коде

**В репозитории:** `runs/session-01.md`, две расшифровки вашего цикла, обычная и с `--break`, одно предложение о разнице.
**Ловушка дня:** слишком малый бюджет токенов выглядит как ошибка парсера; настоящую причину показывают `finish_reason` и `usage`.
**Спросите себя:** почему наблюдение возвращается сообщением `user` и что сломается, если вернуть его как `assistant`?
**В следующий раз:** цикл станет графом, `parse_action` исчезнет, а наблюдение станет сообщением инструмента с id.